# NicheJEPA Tutorial

- Creator: Sebastian Birk (sb75@sanger.ac.uk) Mohammad Vali Sanian (mohammad.sanian@helsinki.fi)
- Date of Creation: 30.06.2025
- Date of Last Modification: 08.07.2025 (by Mohammad Vali Sanian mohammad.sanian@helsinki.fi)

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import logging
import os
import warnings
from typing import List, Tuple
import torch
from sklearn.metrics import silhouette_score
import gseapy as gp

import anndata as ad
import pandas as pd
import scanpy as sc
import scib_metrics
from datasets import load_from_disk
from matplotlib.colors import Normalize
import squidpy as sq
from tqdm import tqdm
from app.infer import (embed_dataset,
                       harmonize_adata,
                       tokenize_adata,
                       perturb_dataset,
                       harmonize_tokenize_embed_pipeline,
                       get_gene_embed,
                       get_average_gene_embed,
                       get_spatial_score,
                       get_emd_distance )
from nichejepa.utils.evaluation import (get_top_gene_score,
                                        get_top_gene_pairs
                                        )
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

In [ ]:
## This function takes a list of cell_ids and returns the corresponding dataset object.
def subset_by_cell_ids(dataset, cell_id_list):
    cell_id_set = set(cell_id_list)
    cell_ids = dataset['cell_id']
    indices = [i for i, cid in tqdm(enumerate(cell_ids), total=len(cell_ids), desc="Finding matching indices") if cid in cell_id_set]
    return dataset.select(indices)


### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/lustre/scratch126/cellgen/lotfollahi/sb75/nichejepa-reproducibility/artifacts'
model_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/27092025_012758_922'

### 1.3 Define Config

In [ ]:
min_cells_per_niche = 100
latent_cluster_key = 'predicted_niche'

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
#sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'
sc.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)


In [ ]:
#os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
#os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)
#os.makedirs(f'{benchmarking_artifacts_folder_path}/NicheJEPA', exist_ok=True)

## 2. Analysis

### 2.1 xhb1002-AT15-10b_1p (GBM)

In [ ]:
dataset_name = 'xhb1002-AT15-10b_1p'
cell_type_key = 'leiden_log1pnorm_res-1'
spot_size = 20
file_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/AT15-BRA-4-FO-G1-S24__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad'

In [ ]:
dataset_name = 'xhk1007-CV1-41b_1p'
cell_type_key = 'level_1_cell_type'
spot_size = 20
file_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/CV1-KID-0-FT-2.h5ad'


In [ ]:
dataset_name = 'xhs1000-39b_1p'
cell_type_key = 'niche_type'
spot_size = 20
file_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch19.h5ad'

#### 2.1.1 Load Data

In [ ]:
adata = sc.read_h5ad(file_path)

In [ ]:
# Visualize data
sc.pl.spatial(adata,
              spot_size=spot_size,
              color=cell_type_key)

In [ ]:
# Visualize data
sc.pl.spatial(adata,
              spot_size=spot_size,
              color=cell_type_key)

#### 2.1.2 Harmonize Data
If you’ve already tokenized your data, go to Load Tokenized Data and load the previously tokenized files.

In [ ]:
adata = harmonize_adata(adata)

#### 2.1.3 Tokenize Data
If you’ve already tokenized your data, go to Load Tokenized Data and load the previously tokenized files.

In [ ]:
dataset = tokenize_adata(adata,
                         model_folder_path,
                         nproc = 4,
                         processing_mode = 'parallel')

#item = dataset[0]
#print(item)

#### 2.1.4 Save Tokenize Data

In [ ]:
num_shards = 32
save_dataset_path = f'/lustre/scratch126/cellgen/lotfollahi/mv10/tokenized_data/{dataset_name}'

In [ ]:
!mkdir save_dataset_path

In [ ]:
dataset.save_to_disk(
            save_dataset_path,
            num_shards=num_shards)

#### 2.1.5 Load Tokenize Data

In [ ]:
f'{artifacts_folder_path}/{dataset_name}/{file_path.split("/")[-1].split(".")[0]}.dataset'

In [ ]:
load_dataset_path = f'/lustre/scratch126/cellgen/lotfollahi/mv10/tokenized_data/{dataset_name}'
dataset = load_from_disk(load_dataset_path)

In [ ]:
temp = dataset['cell_id']

In [ ]:
len(temp)

#### 2.1.6 Embed Data

In [ ]:
emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None
top_k = None

In [ ]:
output_embed = embed_dataset(
    dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
dataset['seg_tokens'][0][:1000]

In [ ]:
adata.obsm['cell_emb'] = output_embed['cell_emb']
adata.obsm['neighborhood_emb'] = output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='neighborhood_emb',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')
# Store the neighborhood UMAP with a custom key
adata.obsm['neighborhood_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='cell_emb',
                key_added='cell')
sc.tl.umap(adata,
           neighbors_key='cell')
# Store the cell UMAP with a custom key
adata.obsm['cell_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='cell',
           color=label)


In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='neighborhood_emb',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')
# Store the neighborhood UMAP with a custom key
adata.obsm['neighborhood_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='cell_emb',
                key_added='cell')
sc.tl.umap(adata,
           neighbors_key='cell')
# Store the cell UMAP with a custom key
adata.obsm['cell_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='cell',
           color=label)


In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='neighborhood_emb',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')
# Store the neighborhood UMAP with a custom key
adata.obsm['neighborhood_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='cell_emb',
                key_added='cell')
sc.tl.umap(adata,
           neighbors_key='cell')
# Store the cell UMAP with a custom key
adata.obsm['cell_umap'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata,
           neighbors_key='cell',
           color=label)


#### 2.1.7 Niche Identification

In [ ]:
# Set dataset params
batch_key = 'batch'
emb_key = 'cell'
latent_leiden_resolution = 0.3
spot_sizes = [0.3, 30]

latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'


In [ ]:
# Compute UMAP representation for (aggregated) gene embeddings
if F'{emb_key}_connectivities' not in adata.obsp.keys():
    print("Computing sc.pp.neighbors...")
    sc.pp.neighbors(adata,
                    n_neighbors=15,
                    use_rep=f'{emb_key}_emb_layer_{layer}',
                    key_added=emb_key)
if f'{emb_key}_umap' not in adata.obsm.keys():
    print("Computing sc.tl.umap...")
    sc.tl.umap(adata,
               neighbors_key=emb_key,
               key_added=f'{emb_key}_umap')


In [ ]:
sc.tl.leiden(adata,
             neighbors_key=emb_key,
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
sc.pl.spatial(
    adata,
    color=latent_cluster_key,
    spot_size=spot_size,
    save=f'{dataset_name}_{latent_cluster_key}.svg'
)

#### get summery of gene_experesion on this data

In [ ]:
def summarize_gene_expression(
    adata,
    subset_cell_ids: list,
    top_n: int = 10,
    ensembl_col: str = "ensembl_id"
):
    """
    Filter AnnData to a subset of cells, compute gene expression summaries,
    and return top expressed genes with Ensembl IDs.

    Args:
        adata (AnnData): Pre-loaded AnnData object.
        subset_cell_ids (list): List of cell IDs to filter.
        top_n (int): Number of top expressed genes to return.
        ensembl_col (str): Column in adata.var that contains Ensembl IDs.

    Returns:
        combined_counts (pd.DataFrame): Filtered cell-by-gene count matrix.
        top_gene_df (pd.DataFrame): DataFrame with top gene names and Ensembl IDs.
    """
    if 'cell_id' not in adata.obs.columns:
        raise KeyError("Expected `adata.obs` to contain a 'cell_id' column.")
    if ensembl_col not in adata.var.columns:
        raise KeyError(f"Expected `adata.var` to contain '{ensembl_col}' column.")

    # Filter to subset of cells
    mask = adata.obs['cell_id'].astype(str).isin(set(subset_cell_ids))
    filtered = adata[mask].copy()
    if filtered.n_obs == 0:
        raise ValueError("No matching cells found in the provided subset.")

    # Extract dense matrix
    matrix = filtered.X.toarray() if hasattr(filtered.X, "toarray") else filtered.X
    combined_counts = pd.DataFrame(matrix, columns=filtered.var_names, index=filtered.obs_names)

    # Compute mean expression and get top N genes
    mean_expr = combined_counts.mean(axis=0)
    top_genes = mean_expr.nlargest(top_n)

    # Map to Ensembl IDs
    var_df = adata.var.loc[top_genes.index]
    top_gene_df = pd.DataFrame({
        "gene_name": top_genes.index,
        "ensembl_id": var_df[ensembl_col].values,
        "mean_expression": top_genes.values
    })

    return combined_counts, top_gene_df


In [ ]:
combined_counts, top_gene_df = summarize_gene_expression(adata, list(adata.obs['cell_id']),top_n=1000)

#### 2.1.8 compute_emd_distance
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
# each list should at least contain 20 gene each 
# to have score that is meaningful
# compute mmd also and add it 
cell_gene_ensembl_id = list(top_gene_df['ensembl_id'])
neighborhood_gene_ensembl_id = list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)


In [ ]:
output_emd_distance = get_emd_distance( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
adata.obs['emd_dist'] = output_emd_distance

In [ ]:
# 2) Normalize your data so that 0→0; 0.6→1.0
norm = Normalize(vmin=0.0, vmax=max(output_emd_distance))

# 3) Build the colormap
#cmap = make_thresholded_cmap(base_cmap="plasma", threshold=0.8)

sq.pl.spatial_scatter(
        adata,
        color='emd_dist',
        #cmap=cmap,
        norm=norm,
        shape=None,
        figsize=(10, 8),
        size=1.5,
        title=f"{dataset_name} - (EMD distance)",
)
plt.show()

#### 2.1.9 subset dataset instruction
You can use this method of subsetting dataasets in all your tasks. This is just a template for how to use it.

In [ ]:
subset_cell_ids = ['1002_batch40_0',
 '1002_batch40_1',
 '1002_batch40_2',
 '1002_batch40_3',
 '1002_batch40_4',
 '1002_batch40_5',
 '1002_batch40_6',
 '1002_batch40_7',
 '1002_batch40_8',
 '1002_batch40_9']


In [ ]:
dataset_subset = subset_by_cell_ids(dataset, subset_cell_ids)

In [ ]:
dataset_subset

#### 2.1.10 get_gene_representation
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
emb_layer = None
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12

In [ ]:
cell_gene_ensembl_id = ['ENSG00000104419', 'ENSG00000130402']
neighborhood_gene_ensembl_id = ['ENSG00000104419','ENSG00000130402'] # ['all'] * len(perturbed_cell_id)

In [ ]:
output_gene_embed = get_gene_embed( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
for gene_ens in output_gene_embed.keys():
    adata.obsm[gene_ens] = output_gene_embed[gene_ens]

#### 2.1.11 get_average_gene_representation
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
list(adata.var['ensembl_id'])

In [ ]:
cell_gene_ensembl_id = list(top_gene_df['ensembl_id'])
neighborhood_gene_ensembl_id = list(top_gene_df['ensembl_id']) # ['all'] * len(perturbed_cell_id)

cell_gene_ensembl_id = list(adata.var['ensembl_id'])
neighborhood_gene_ensembl_id = list(adata.var['ensembl_id'])

In [ ]:
output_average_gene_embed = get_average_gene_embed(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
adata.uns['cell_gene_emb_average_per_data'] = output_average_gene_embed['cell_gene_emb_average_per_data']
adata.uns['cell_gene_emb_counts_per_data'] = output_average_gene_embed['cell_gene_emb_counts_per_data']
adata.uns['neighborhood_gene_emb_average_per_data'] = output_average_gene_embed['neighborhood_gene_emb_average_per_data']
adata.uns['neighborhood_gene_emb_counts_per_data'] = output_average_gene_embed['neighborhood_gene_emb_counts_per_data']

In [ ]:
adata.uns['cell_gene_emb_average_per_data']

In [ ]:
# 1) You can choose any of the embeddings, which could be either a cell or a neighborhood.
gene_emb_name = 'neighborhood_gene_emb_average_per_data'
gene_emb_name = 'cell_gene_emb_average_per_data'


In [ ]:
gene_emb_name

In [ ]:
gene_emb = np.array(adata.uns[gene_emb_name])  # (n_genes, emb_dim)
gene_ids =  np.array(cell_gene_ensembl_id,  dtype=str)                  # (n_genes,)

# 2) Remove any gene rows containing NaNs
mask = ~np.isnan(gene_emb).any(axis=1)
gene_emb_clean = gene_emb[mask]
gene_ids_clean = gene_ids[mask]

# 3) Build a small AnnData where each "obs" is one gene
adata_genes = ad.AnnData(
    X=gene_emb_clean,
    obs=pd.DataFrame({'gene_id': gene_ids_clean})
)
adata_genes.obs_names = gene_ids_clean.astype(str)

# 4) Compute a neighborhood graph on the embeddings
sc.pp.neighbors(
    adata_genes,
    use_rep='X',
    n_neighbors=15,
    metric='euclidean'
)

# 5) Run UMAP
sc.tl.umap(
    adata_genes,
    #min_dist=0.5,
    #spread=1.0
)


# 7) (Optional) Quick plot
sc.pl.umap(
    adata_genes,
    title='Gene embedding UMAP',
    show=True
)


In [ ]:
sc.tl.pca(adata_genes, n_comps=2, use_highly_variable=False, svd_solver='arpack')

sc.pl.pca(
    adata_genes,
    components=['1,2'],   # plot PC1 vs PC2
    color=None,
    title='Gene embeddings PCA (PC1 vs PC2)',
    show=True
)


In [ ]:
# Store results here
results = []

# Define parameters
knn_values = [5, 10, 15, 20]
resolutions = np.arange(0.1, 1.1, 0.1)  # 0.1 to 1.0

# Loop over each KNN and resolution combination
for knn in knn_values:
    sc.pp.neighbors(adata_genes, n_neighbors=knn, use_rep='X')  # use 'X' or other embedding like 'X_pca'
    
    for res in resolutions:
        # Run Leiden clustering
        sc.tl.leiden(adata_genes, resolution=res, key_added='leiden_tmp')
        
        # Compute silhouette score
        labels = adata_genes.obs['leiden_tmp']
        if len(np.unique(labels)) < 2:
            score = np.nan  # silhouette score not defined for 1 cluster
        else:
            score = silhouette_score(adata_genes.obsm['X_pca'], labels)  # you can change to 'X' if no PCA
        
        # Store results
        results.append({
            'knn': knn,
            'resolution': round(res, 2),
            'silhouette_score': score
        })

# Convert results to DataFrame
df = pd.DataFrame(results)
df['combo'] = df['knn'].astype(str) + '_k_' + df['resolution'].astype(str)

# Plot
plt.figure(figsize=(12, 6))
df_sorted = df.sort_values('silhouette_score', ascending=False)
plt.bar(df_sorted['combo'], df_sorted['silhouette_score'], color='skyblue')
plt.xticks(rotation=90)
plt.ylabel("Silhouette Score")
plt.xlabel("KNN_Resolution Combination")
plt.title("Silhouette Score for Different KNN and Leiden Resolution Settings")
plt.tight_layout()
plt.show()


In [ ]:
sc.pp.neighbors(adata_genes,n_neighbors=20)
sc.tl.leiden(adata_genes,resolution=0.3,key_added='r9')

In [ ]:
sc.pl.umap(adata_genes,color='r9',frameon=False)

In [ ]:
id_to_name = {ensg: name for name, ensg in adata.var['ensembl_id'].items()}

In [ ]:
interesting_group = '10'
ens_interesting_group = list(adata_genes.obs[adata_genes.obs['r9']==interesting_group]['gene_id'])

In [ ]:
genes  = [ id_to_name.get(ensg, "UNKNOWN") for ensg in ens_interesting_group]

In [ ]:
len(genes)

In [ ]:
# 2) Your input gene list
# 3) Call gseapy.enrichr
#    You can choose any Enrichr library; here are some examples:
# I should create a mapping from gene_id to ens
# from 110 release
libraries = [
    'KEGG_2019_Human',
    'GO_Biological_Process_2021',
    'Reactome_Pathways_2024',
    'MSigDB_Hallmark_2020'
]

# Run ORA for each library
results = {}
for lib in libraries:
    enr = gp.enrichr(
        gene_list = genes,
        gene_sets  = lib,
        outdir     = None,        # don't write files
        cutoff     = 0.05,         # only show terms with adjusted p < 0.05
    )
    # enr.results is a pandas DataFrame
    results[lib] = enr.results

# 4) Inspect top enriched terms for each library
for lib, df in results.items():
    print(f"\n=== Top 5 terms in {lib} ===")
    display(df[['Term','Adjusted P-value','Combined Score']].head(20))


#### 2.1.12 compute_spatial_score
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
adata.var

In [ ]:
# for out put of get_top_gene_score be correct both cell_gene_ensembl_id and 
# neighborhood_gene_ensembl_id should have same size and same order
cell_gene_ensembl_id =  list(top_gene_df['ensembl_id']) 
neighborhood_gene_ensembl_id =  list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)

In [ ]:
output_cosine_sim_score = get_spatial_score(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
gene_pair_score = output_cosine_sim_score['cos_sim_neighborhood'] / output_cosine_sim_score['cos_sim_cell']

In [ ]:
gene_pair_score

In [ ]:
output_cosine_sim_score['cell_count_neighborhood']

In [ ]:
df_gene_scores = get_top_gene_score(gene_pair_score, 
                                    cell_gene_ensembl_id, 
                                    gene_df=adata.var, 
                                    gene_counts= output_cosine_sim_score['cell_count_neighborhood'],
                                    min_count = 20)

In [ ]:
df_gene_scores.sort_values(by='gene_score', ascending=False)

In [ ]:
df_gene_pairs_score = get_top_gene_pairs(
    torch.from_numpy(gene_pair_score),
    count_cell=torch.from_numpy(output_cosine_sim_score['cell_count_neighborhood']),
    count_neb=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_cell=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_neb=torch.from_numpy(output_cosine_sim_score['cos_sim_neighborhood']),
    gene_df=adata.var,
    cell_gene_ids=cell_gene_ensembl_id,
    neighborhood_gene_ids=neighborhood_gene_ensembl_id,
    min_count=20,
    sim_thresh=0.0,
    k=500
)


In [ ]:
df_gene_pairs_score

#### 2.1.13 Perturbed Data

In [ ]:
# Define perturbations
perturbed_cell_id = ['1007_batch20_0', '1007_batch20_0', '1007_batch20_1']
perturbed_ensembl_id = ['all', 'all', 'ENSG00000169194'] # ['all'] * len(perturbed_cell_id)
perturbation_target = ['cell', 'neighborhood', 'cell'] # ['neighborhood'] * len(perturbed_cell_id)
perturbation_type = ['knockout', 'knockout', 'foldchange'] # ['knockout'] * len(perturbed_cell_id)
foldchange = [np.nan, np.nan, 0.5] # [np.nan] * len(perturbed_cell_id)

perturb_df =  pd.DataFrame({
    'perturbed_cell_id': perturbed_cell_id,
    'perturbed_ensembl_id': perturbed_ensembl_id,
    'perturbation_target': perturbation_target,
    'perturbation_type': perturbation_type, 
    'foldchange': foldchange
})

In [ ]:
perturbed_dataset = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    model_folder_path=model_folder_path)

#### 2.1.14 Embed Perturb Data

In [ ]:
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    cell_gene_ids=cell_gene_ids,
    neighborhood_gene_ids=neighborhood_gene_ids,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
adata.obsm['perturbed_cell_emb'] = perturbed_output_embed['cell_emb']
adata.obsm['perturbed_neighborhood_emb'] = perturbed_output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_neighborhood_emb',
                key_added='perturbed_neighborhood')
sc.tl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           key_added='perturbed_neighborhood_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_neighborhood_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_cell_emb',
                key_added='perturbed_cell')
sc.tl.umap(adata,
           neighbors_key='perturbed_cell',
           key_added='perturbed_cell_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_cell_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_cell',
           color=label)